# Regular Expressions

A regular expression (shortened as regex or regexp), sometimes referred to as a rational expression, is a sequence of characters that specifies a match pattern in text. Usually such patterns are used by string-searching algorithms for "find" or "find and replace" operations on strings, or for input validation. Regular expression techniques are developed in theoretical computer science and formal language theory.

In [9]:
import re
from typing import Self, Optional
from dataclasses import dataclass, field
import colorama

res = re.search("seminars?k?i?","seminar")
if res:
    print("Found seminar in expression: 'seminars?k?i?'")
else:
    print("Not found.")

Found seminar in expression: 'seminars?k?i?'


# Types of Regex engines
There are two types of Regex engines:
1. Regex-directed engines ( Nondeterministic Finite Automaton engines )
2. Text-directed engines  ( Deterministic Finite Automaton engines )

There are pros and cons to each model, the main ones being that NFA engines support more complex patterns, but can run in O(2^n) in worst case scenarios, and that DFA is strictly linear in execution time (O(n)).

In this project, we will implement a NFA Regex engine.


In [10]:
class StateContext():
    text = None
    index = 0
    captured_groups = [] # TODO: Finish implementation
    
    
    def __init__(self,text,index=0,captured_groups=[]):
        self.text = text
        self.index = index
        self.captured_groups = captured_groups
        self.trace = []
        pass

In [11]:
from typing import Self, Optional
from dataclasses import dataclass

@dataclass
class Node():
    def __init__(self, val):
        self.val = val
        self.children = []
        
    def accept(self, visitor): # TODO: Hook this up for captured groups
        method_name = f'visit_{self.__class__.__name__.lower()}'
        visitor_method = getattr(visitor, method_name, visitor.generic_visit())
        return visitor_method(self)
    
    def match(self, ctx: 'StateContext', remaining: list[Self]):
        pass

@dataclass
class Literal(Node):
    char: str
    regex_idx: int = -1
    
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        ctx.trace.append((self.regex_idx, ctx.index, "eval"))
        
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] == self.char:
            ctx.trace.append((self.regex_idx, ctx.index, "match"))
            ctx.index += 1 # Consume
            
            # If there's more regex, try to match it
            if not remaining or remaining[0].match(ctx, remaining[1:]):
                return True
                
            # If the REST of the regex failed, we must backtrack this character
            ctx.index -= 1 
            ctx.trace.append((self.regex_idx, ctx.index, "backtrack"))
            return False
        else:
            ctx.trace.append((self.regex_idx, ctx.index, "fail"))
            return False

@dataclass
class Concatenation(Node):
    children: list[Node]
    regex_idx: int = -1
    
    def match(self, ctx: StateContext, remaining: list[Node] = None) -> bool:
        if not self.children:
            return True
            
        all_remaining = self.children + (remaining or [])
        return all_remaining[0].match(ctx, all_remaining[1:])
    
@dataclass
class Alternation(Node):
    left: Node
    right: Node
    regex_idx: int = -1
    
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        start_index = ctx.index
        ctx.trace.append((self.regex_idx, ctx.index, "eval"))
        
        # Try Left Branch
        if self.left.match(ctx, remaining):
            return True
        
        # Left failed, reset and try Right Branch
        ctx.index = start_index
        ctx.trace.append((self.regex_idx, ctx.index, "backtrack")) 
        return self.right.match(ctx, remaining)

@dataclass
class Wildcard(Node):
    regex_idx: int = -1
    
    def match(self, ctx: StateContext, remaining: list[Node]) -> bool:
        ctx.trace.append((self.regex_idx, ctx.index, "eval"))
        
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] != '\n':
            ctx.trace.append((self.regex_idx, ctx.index, "match"))
            ctx.index += 1 
            
            if not remaining or remaining[0].match(ctx, remaining[1:]):
                return True
                
            ctx.index -= 1 # Backtrack
            ctx.trace.append((self.regex_idx, ctx.index, "backtrack"))
        else:
            ctx.trace.append((self.regex_idx, ctx.index, "fail"))
            
        return False

@dataclass
class CharClass(Node):
    def __init__(self, val: str, regex_idx: int = -1):
        self.val = val
        self.regex_idx = regex_idx
        self.chars = set()
        
        i = 0
        while i < len(val):
            if i + 2 < len(val) and val[i+1] == "-":
                start_char = val[i]
                end_char = val[i+2]
                
                for c in range(ord(start_char), ord(end_char) + 1):
                    self.chars.add(chr(c))
                i += 3
            else:
                self.chars.add(val[i])
                i += 1
        
    def match(self, ctx: StateContext, remaining: list[Node]):
        ctx.trace.append((self.regex_idx, ctx.index, "eval"))
        
        if ctx.index < len(ctx.text) and ctx.text[ctx.index] in self.chars:
            ctx.trace.append((self.regex_idx, ctx.index, "match"))
            ctx.index += 1
            
            if not remaining or remaining[0].match(ctx, remaining[1:]):
                return True
                
            ctx.index -= 1
            ctx.trace.append((self.regex_idx, ctx.index, "backtrack"))
        else:
            ctx.trace.append((self.regex_idx, ctx.index, "fail"))
            
        return False

@dataclass
class Quantifier(Node):
    child: Node
    min_repeat: int
    max_repeat: Optional[int]
    regex_idx: int = -1
    
    def match(self, ctx: 'StateContext', remaining: list[Node]) -> bool:
        start_index = ctx.index
        match_indices = [start_index]
        
        # 1. Greedy Phase: Match as much as possible
        while self.max_repeat is None or (len(match_indices) - 1) < self.max_repeat:
            # We pass an empty list because the quantifier handles the 'remaining' nodes itself later
            if self.child.match(ctx, []): 
                match_indices.append(ctx.index)
            else:
                break

        # 2. Backtrack Phase: Start from the longest match and shrink
        while len(match_indices) - 1 >= self.min_repeat:
            current_match_end = match_indices.pop()
            ctx.index = current_match_end
            
            # Log that the Quantifier is now 'evaluating' this specific branch length
            ctx.trace.append((self.regex_idx, ctx.index, "eval"))
            
            if not remaining:
                # If nothing is left in the regex, the quantifier is happy with this match
                ctx.trace.append((self.regex_idx, ctx.index, "match"))
                return True
            
            # Try to match the rest of the regex string after this quantifier
            if remaining[0].match(ctx, remaining[1:]):
                return True
            
            # If the rest failed, and we have more match_indices, we will 'backtrack' in the next loop
            if match_indices:
                ctx.trace.append((self.regex_idx, ctx.index, "backtrack"))
        
        # If no repetition length worked
        ctx.index = start_index
        ctx.trace.append((self.regex_idx, ctx.index, "fail"))
        return False
    
@dataclass
class Group(Node): # TODO: use 
    child: Node
    index: int
    is_capturing: bool
    regex_idx: int = -1
    
@dataclass
class End(Node):
    def match(self, ctx: 'StateContext', remaining: list[Node] = None) -> bool:
        return ctx.index == len(ctx.text)
    
def generate_mermaid(root: Node, title: str = None) -> str:
    """Traverses the regex AST and generates a Mermaid.js flowchart string."""
    lines = ["```mermaid"]
    
    if title:
        lines.append("---")
        lines.append(f"title: {title}")
        lines.append("---")
        
    lines.append("graph TD")
    
    def escape(text):
        safe_text = str(text).replace('"', '\\"')
        return f'"{safe_text}"'

    def traverse(node: Node) -> str:
        node_id = f"node_{id(node)}"
        label = ""
        children = []
        
        if isinstance(node, Literal):
            label = f"Literal: '{node.char}'"
        elif isinstance(node, Concatenation):
            label = "Concatenation"
            children = node.children
        elif isinstance(node, Alternation):
            label = "Alternation (|)"
            children = [node.left, node.right]
        elif isinstance(node, Wildcard):
            label = "Wildcard (.)"
        elif isinstance(node, CharClass):
            label = f"CharClass: {node.val}"
        elif isinstance(node, Quantifier):
            max_rep = "∞" if node.max_repeat is None else node.max_repeat
            label = f"Quantifier ({node.min_repeat} to {max_rep})"
            children = [node.child]
        elif isinstance(node, Group):
            label = f"Group #{node.index}"
            children = [node.child]
        elif isinstance(node, End):
            label = "End"
        else:
            label = type(node).__name__
            
        lines.append(f"    {node_id}[{escape(label)}]")
        
        for child in children:
            if child:
                child_id = traverse(child)
                lines.append(f"    {node_id} --> {child_id}")
                
        return node_id

    traverse(root)
    lines.append("```")
    return "\n".join(lines)

In [12]:
class Tester():
    tests: dict[str, list[tuple[Node,str, bool]]]
    rule_length = 0
    def __init__(self,raw_tests):
        self.tests = {}
        for rule,tree,string,expected in raw_tests:
            self.rule_length = max(self.rule_length,len(rule)+4)
            if rule in self.tests.keys():
                self.tests[rule].append((tree,string,expected))
            else:
                self.tests[rule] = [(tree,string,expected)]
        pass
    
    def test(self, num=-1):
        i = 0
        res = []

        for rule_name, tests in self.tests.items():
            for (tree,string,answer) in tests:
                if num==-1 or num==i:
                    if tree.match(StateContext(string)) == answer:
                        # Number, expected, Correct, String
                        res.append([i,answer,True, string,rule_name])
                    else:
                        res.append([i,not answer,False, string,rule_name])
                i+=1
        r_l = self.rule_length
        for r in res:
            print(f"Test {str(r[0]):<4}| "
            f"Pass: {colorama.Fore.GREEN if r[2] else colorama.Fore.RED}{'PASS' if r[2] else 'FAIL':<6}{colorama.Style.RESET_ALL}| "
            f"Rule: /\"{(r[4]+f"\"/"):<{r_l}}|"
            f"Str: {str(r[3]):<15}| "
            f"Res: {colorama.Fore.GREEN if r[1] else colorama.Fore.RED}{str(r[1]):<6}{colorama.Style.RESET_ALL}|")

# Lexer
A lexer is an automaton, in this case, a **Nondeterministic Finite Automaton**. An automaton is an abstract mathematical model of a computing machine. It only knows how to read a sequence of inputs, and move between different states based on predefined rules.

A finite state automation in mathematics is defined by five components:  (**Q**, **Σ**, **δ**, **q₀**, **F**): 

**Q**: Finite set of all possible states

**Σ** (Sigma): A finite set of input symbols (the alphabet).

**δ** (Delta): The transition function that maps a state and an input symbol to the next state. This is 
the set of rules dictating how the machine moves from one state to another.

**q₀**: The initial state where the process starts.

**F**: A set of final or accepting states.

---

# Determinism vs. Nondeterminism
1. DFA (**Deterministic Finite Automaton**) - For any given state and any given input, there is excactly one possible next state.
2. NFA (**Nondeterministic Finite Automaton**) - For a given state and input, there can be multiple valid next states, which must be explored until a working one is found.

In [13]:
@dataclass
class Token:
    type: str  
    value: str 
    index : int = -1

class Lexer:
    def __init__(self, text: str):
        self.text = text

    def tokenize(self) -> list[Token]:
        tokens = []
        i = 0
        while i < len(self.text):
            char = self.text[i]
            
            if char == '*':
                tokens.append(Token('STAR', char,i))
            elif char == "?":
                tokens.append(Token('QMARK',char,i))
            elif char == "+":
                tokens.append(Token("PLUS",char,i))
            elif char == '.':
                tokens.append(Token("DOT", char,i))
            elif char == '|':
                tokens.append(Token("PIPE",char,i))
            elif char == '[':
                start_i = i
                i += 1
                class_content = ""
                # Consume everything until the closing bracket
                while i < len(self.text) and self.text[i] != ']':
                    class_content += self.text[i]
                    i += 1
                
                if i >= len(self.text):
                    raise ValueError("Unclosed character class '['")

                tokens.append(Token("CHAR_CLASS", class_content,start_i))
            elif char == '(':
                tokens.append(Token("LPAREN",char,i))
            elif char == ')':
                tokens.append(Token("RPAREN",char,i))
            elif char == '\\':
                if i + 1 < len(self.text):
                    tokens.append(Token('CHAR', self.text[i+1],i))
                    i += 1
                else:
                    raise ValueError("Unused escape char at end of string")
            else:
                tokens.append(Token('CHAR', char,i))
            i += 1
        return tokens

# Parsing Regex

In regex, the precendence from highest to lowest is:
1. Parentheses / Base units: literals, wildcards, etc.
2. Quantifiers: **\***, **?**, **+**
3. Concatenation: **abc**
4. Alternation **a|b**

To implement this, we work our way up from the bottom.
We start with ```parse_expression```, which handles the ```OR``` ( **|** ). It then asks ```parse_term``` and ```parse_expression``` to recursively evaluate the left and right side, respectively.

We then go to ```parse_term```, which then loops and keeps handling chunks until it hits something that breaks a chain ( like the end of the string, or a pipe).

The next layer down is ```parse_factor```, which grabs the current node, and then peeks to see if there is a quantifier behind it. If there is one, it wraps the atom in the corresponding quantifier and continues down.

The final layer is ```parse_atom```, which handles the literals, wild-cards, and sub-expressions, in which case it goes back to the top, with ``````parse_expression``````.

In [14]:
class Parser:
    def __init__(self, tokens: list[Token]):
        self.tokens = tokens
        self.pos = 0

    def peek(self) -> Token | None:
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return None
    
    def consume(self, expected_type: str) -> Token | None:
        token = self.peek()
        if not token or token.type != expected_type:
            raise ValueError(f"Expected {expected_type}, got {token.type if token else None}")
        self.pos += 1
        return token

    def parse_atom(self) -> Node:
        token = self.peek()
        if not token:
            raise ValueError("Unexpected end of input")
    
        if token.type == "CHAR":
            self.consume("CHAR")
            # Pass the token's index to Literal
            return Literal(token.value, regex_idx=token.index)

        elif token.type == "CHAR_CLASS":
            self.consume("CHAR_CLASS")
            # Pass the token's index to CharClass
            return CharClass(token.value, regex_idx=token.index)

        elif token.type == "DOT":
            self.consume("DOT")
            # Pass the token's index to Wildcard
            return Wildcard(regex_idx=token.index)
        
        elif token.type == "LPAREN":
            self.consume("LPAREN")
            node = self.parse_expression()
            self.consume("RPAREN")
            return node
        
        else:
            raise ValueError(f"Unexpected token @ index: {self.pos} : {token.type}")
            
    def parse_factor(self) -> Node:
        node = self.parse_atom()
        
        token = self.peek()
        if token:
            if token.type == "STAR":
                self.consume("STAR")
                # Pass the token's index to Quantifier
                return Quantifier(child=node, min_repeat=0, max_repeat=None, regex_idx=token.index)

            elif token.type == "QMARK":
                self.consume("QMARK")
                # Pass the token's index to Quantifier
                return Quantifier(child=node, min_repeat=0, max_repeat=1, regex_idx=token.index)
            
            elif token.type == "PLUS":
                self.consume("PLUS")
                # Pass the token's index to Quantifier
                return Quantifier(child=node, min_repeat=1, max_repeat=None, regex_idx=token.index)
                
        return node

    def parse_expression(self) -> Node:
        left = self.parse_term()
        
        token = self.peek()
        if token and token.type == "PIPE":
            self.consume("PIPE")
            right = self.parse_expression()
            # Pass the token's index to Alternation
            return Alternation(left, right, regex_idx=token.index)

        return left
    
    def parse_term(self) -> Node:
        nodes = []
        while self.peek() and self.peek().type not in ["PIPE", "RPAREN"]:
            nodes.append(self.parse_factor())
        
        if not nodes:
            return Concatenation([])

        if len(nodes) == 1:
            return nodes[0]
        
        return Concatenation(nodes)
        

    def parse(self) -> Node:
        root_node = self.parse_expression()
        return Concatenation([root_node, End()])

# Example execution
Let's go over how a regular expression gets parsed.
Example regex: `(abc|def)+_..._x*y?`

1. Enter `parse_expression`
2. Nothing in left `parse_term`
3. Parse expression starting at index 1
4. Re-enter `parse_term`, cover "abc" literal, hit pipe
5. Hit pipe inside `parse_expression`
6. Enter `parse_expression` to the right of the pipe
7. Go until right paren, cover "def" literal.
8. Go back twice
9. Now have an `Alternation` node, of (`Concat("a","b","c")`, `Concat("d","e","f")`)
10. Which can happen 1 or more times due to the `+`.
11. Then we match the `_`, the three `.`s and the `_`.
12. We then match the x 0 or more times, due to the `*`, and then match the y 0 or 1 times, due to the `?`.

---

Following this logic, `abcabcabc_xyz_xx` for example, will get matched.
Let's check in the cell below.

In [15]:
lexer = Lexer(r"(abc|def)+_..._x*y?")
tokens = lexer.tokenize()
parser = Parser(tokens)
root = parser.parse()
ctx = StateContext("abcdefabc_xyz_xx")
matched = root.match(ctx)
if matched:
    print("Match found")
else:
    print("Given string does not match the regular expression")
#lexer = Lexer(r"(reg)*-?(ex)+")
#tokens = lexer.tokenize()
#parser = Parser(tokens)
#root = parser.parse()
#mermaid_code = generate_mermaid(root,"AST structure for \"(reg)*-?(ex)+\"")
#print(mermaid_code)


Match found


# Testing
Below is the cell responsible for running unit tests, to notice if something breaks when a change is made.

In [16]:
lexer_1 = Lexer(r"ab.d_*\.J?")
tokens_1 = lexer_1.tokenize()
parser_1 = Parser(tokens_1)
root_1 = parser_1.parse()

lexer_2 = Lexer(r"a*a")
tokens_2 = lexer_2.tokenize()
parser_2 = Parser(tokens_2)
root_2 = parser_2.parse()

lexer_3 = Lexer(r"a*b")
tokens_3 = lexer_3.tokenize()
parser_3 = Parser(tokens_3)
root_3 = parser_3.parse()

lexer_4 = Lexer(r"a...b")
tokens_4 = lexer_4.tokenize()
parser_4 = Parser(tokens_4)
root_4 = parser_4.parse()

lexer_5 = Lexer(r"a+b")
tokens_5 = lexer_5.tokenize()
parser_5 = Parser(tokens_5)
root_5 = parser_5.parse()

lexer_6 = Lexer(r"cat|dog")
tokens_6 = lexer_6.tokenize()
parser_6 = Parser(tokens_6)
root_6 = parser_6.parse()

lexer_7 = Lexer(r"[a-zA-Z1-5]+_*")
tokens_7 = lexer_7.tokenize()
parser_7 = Parser(tokens_7)
root_7 = parser_7.parse()

tests2 = [
  # misc tests
  [r"ab.d_*\.J?",root_1,"abcd___.J",True],
  [r"ab.d_*\.J?",root_1,"abcd_.J",True],
  [r"ab.d_*\.J?",root_1,"ab--d.J",False],
  [r"ab.d_*\.J?",root_1,"abcd___.",True],
  [r"ab.d_*\.J?",root_1,"ab_x_d___.J",False],
  [r"ab.d_*\.J?",root_1,"abcd.J",True],
  [r"ab.d_*\.J?",root_1,"abxd.",True],
  # a*a tests:
  [r"a*a",root_2,"aa",True],
  [r"a*a",root_2,"aaa",True],
  [r"a*a",root_2,"a",True],
  [r"a*a",root_2,"aba",False],
  [r"a*a",root_2,"abbb",False],
  # a*b tests:
  [r"a*b",root_3,"ab",True],
  [r"a*b",root_3,"abb",False],
  [r"a*b",root_3,"b",True],
  [r"a*b",root_3,"bb",False],
  # a...b tests:
  [r"a...b",root_4,"a_x_b",True],
  [r"a...b",root_4,"a__b",False],
  [r"a...b",root_4,"a_.x._b",False],
  [r"a...b",root_4,"aaxbb",True],
  # a+b tests:
  [r"a+b",root_5,"ab",True],
  [r"a+b",root_5,"b",False],
  [r"a+b",root_5,"aaaab",True],
  # cat | dog tests
  [r"cat|dog",root_6,"cat",True],
  [r"dog|cat",root_6,"dog|c",False],
  [r"dog|cat",root_6,"d|cat",False],
  [r"cat|dog",root_6,"dog",True],
  [r"cat|dog",root_6,"ca",False],
  [r"cat|dog",root_6,"do",False],
  [r"cat|dog",root_6,"catdog",False],
  # Char class tests
  [r"[a-zA-Z1-5]+_*",root_7,"a",True],
  [r"[a-zA-Z1-5]+_*",root_7,"abcZ12345",True],
  [r"[a-zA-Z1-5]+_*",root_7,"6",False],
  [r"[a-zA-Z1-5]+_*",root_7,"aZ123______",True],
  [r"[a-zA-Z1-5]+_*",root_7,"()",False],
  [r"[a-zA-Z1-5]+_*",root_7,"AZ6___",False],
]

tester = Tester(tests2)
tester.test()

Test 0   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: abcd___.J      | Res: True  |
Test 1   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: abcd_.J        | Res: True  |
Test 2   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: ab--d.J        | Res: False |
Test 3   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: abcd___.       | Res: True  |
Test 4   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: ab_x_d___.J    | Res: False |
Test 5   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: abcd.J         | Res: True  |
Test 6   | Pass: PASS  | Rule: /"ab.d_*\.J?"/      |Str: abxd.          | Res: True  |
Test 7   | Pass: PASS  | Rule: /"a*a"/             |Str: aa             | Res: True  |
Test 8   | Pass: PASS  | Rule: /"a*a"/             |Str: aaa            | Res: True  |
Test 9   | Pass: PASS  | Rule: /"a*a"/             |Str: a              | Res: True  |
Test 10  | Pass: PASS  | Rule: /"a*a"/             |Str: aba            | Res: False |
Test 11  | Pass: PASS  | Rule: /"a*a"/     